# Week 6 — Gadget Pricer (SamuelAdebodun)

**The Price is Right** capstone for **gadgets**: iPhone ecosystem, Samsung, Huawei, and other devices.

- Predict product price from description using **OpenAI**, **Anthropic (ANTHROPIC_API_KEY)**, or **Ollama (local)**.
- Embedded dataset of (description, price) for phones, tablets, wearables, earbuds.
- Evaluate with MAE; optional JSONL; Gradio UI with model dropdown.

In [ ]:
import os
import re
import json
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

if openai_key:
    print(f"OpenAI key loaded (starts with {openai_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set")
if anthropic_key:
    print(f"Anthropic key loaded (starts with {anthropic_key[:7]}...)")
else:
    print("ANTHROPIC_API_KEY not set")

MODEL_OPENAI = "gpt-4o-mini"
MODEL_ANTHROPIC = "claude-3-5-sonnet-20241022"
MODEL_OLLAMA = "llama3.2"

openai_client = OpenAI() if openai_key else None
anthropic_client = anthropic.Anthropic() if anthropic_key else None
ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
print("Clients ready (Ollama: run 'ollama run llama3.2' locally).")

In [ ]:
@dataclass
class Product:
    summary: str
    price: float

# Gadget dataset: iPhone, Samsung, Huawei, Google, wearables, earbuds (description -> price in USD)
GADGETS = [
    Product("iPhone 15 Pro 256GB, Titanium, A17 Pro, 48MP camera", 999.00),
    Product("iPhone 15 128GB, Dynamic Island, USB-C", 799.00),
    Product("iPhone 14 128GB, 6.1 inch Super Retina XDR", 699.00),
    Product("Samsung Galaxy S24 Ultra 256GB, S Pen, 200MP camera", 1299.99),
    Product("Samsung Galaxy S24+ 256GB, 6.7 inch Dynamic AMOLED", 999.99),
    Product("Samsung Galaxy Z Flip 6 256GB, clamshell foldable", 1099.99),
    Product("Huawei P60 Pro 256GB, variable aperture camera", 899.00),
    Product("Huawei MatePad Pro 12.6 inch, HarmonyOS, OLED", 649.00),
    Product("Google Pixel 8 Pro 128GB, Tensor G3, 50MP", 999.00),
    Product("Google Pixel 8 128GB, 6.2 inch OLED", 699.00),
    Product("OnePlus 12 256GB, Snapdragon 8 Gen 3, 100W charging", 799.99),
    Product("Apple AirPods Pro 2nd gen, active noise cancellation, MagSafe", 249.00),
    Product("Apple AirPods 3rd gen, spatial audio, MagSafe charging", 169.00),
    Product("Samsung Galaxy Buds2 Pro, ANC, 360 audio", 229.99),
    Product("Apple Watch Ultra 2, 49mm, titanium, dual-frequency GPS", 799.00),
    Product("Apple Watch Series 9 45mm GPS, S9 chip", 429.00),
    Product("Samsung Galaxy Watch 6 Classic 47mm, Wear OS", 369.99),
    Product("iPad Air 11 inch M2 128GB WiFi", 599.00),
    Product("iPad Pro 12.9 inch M2 256GB WiFi, Liquid Retina XDR", 1099.00),
    Product("Samsung Galaxy Tab S9+ 12.4 inch 256GB, S Pen included", 919.99),
    Product("MacBook Air 13 inch M3 256GB 8GB RAM", 1099.00),
    Product("Samsung 990 Pro 2TB NVMe SSD", 169.99),
    Product("Anker PowerCore 26800mAh portable charger, 3 ports", 59.99),
    Product("Belkin BoostCharge Pro 3-in-1 MagSafe stand", 149.95),
]

def train_test_split(products, test_size=6, seed=42):
    import random
    rng = random.Random(seed)
    shuffled = list(products)
    rng.shuffle(shuffled)
    test = shuffled[:test_size]
    train = shuffled[test_size:]
    return train, test

train_products, test_products = train_test_split(GADGETS)
print(f"Train: {len(train_products)}, Test: {len(test_products)}")

In [ ]:
def messages_for(product):
    msg = f"Estimate the price in USD of this gadget. Respond with the price only, no explanation.\n\n{product.summary}"
    return [{"role": "user", "content": msg}]

def parse_price(value):
    if isinstance(value, (int, float)):
        return float(value)
    s = str(value).replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0

def predict_price(product, model_choice="OpenAI (gpt-4o-mini)"):
    msgs = messages_for(product)
    user_content = msgs[0]["content"]
    try:
        if model_choice == "OpenAI (gpt-4o-mini)":
            if not openai_client:
                return 0.0
            resp = openai_client.chat.completions.create(
                model=MODEL_OPENAI,
                messages=msgs,
                temperature=0,
            )
            return parse_price(resp.choices[0].message.content or "0")
        elif model_choice == "Anthropic (claude-3-5-sonnet)":
            if not anthropic_client:
                return 0.0
            msg = anthropic_client.messages.create(
                model=MODEL_ANTHROPIC,
                max_tokens=64,
                messages=[{"role": "user", "content": user_content}],
            )
            text = msg.content[0].text if msg.content else "0"
            return parse_price(text)
        else:
            resp = ollama_client.chat.completions.create(
                model=MODEL_OLLAMA,
                messages=msgs,
                temperature=0,
            )
            return parse_price(resp.choices[0].message.content or "0")
    except Exception:
        return 0.0

def evaluate_mae(products, model_choice="OpenAI (gpt-4o-mini)"):
    errors = []
    for p in products:
        pred = predict_price(p, model_choice)
        errors.append(abs(pred - p.price))
    return sum(errors) / len(errors) if errors else 0.0

print("Predictor ready. Run the next cell to compute test MAE.")

In [ ]:
# Evaluate on test set with OpenAI (uses API calls). Change model_choice to use another model.
test_mae = evaluate_mae(test_products, model_choice="OpenAI (gpt-4o-mini)")
print(f"Test MAE: ${test_mae:.2f}")

In [ ]:
# Optional: build JSONL for fine-tuning (Day 5 style)
def make_jsonl(products):
    lines = []
    for p in products:
        messages = [
            {"role": "user", "content": f"Estimate the price in USD of this gadget. Respond with the price only.\n\n{p.summary}"},
            {"role": "assistant", "content": f"${p.price:.2f}"},
        ]
        lines.append(json.dumps({"messages": messages}))
    return "\n".join(lines)

jsonl_preview = make_jsonl(train_products[:2])
print("JSONL preview (first 2 items):")
print(jsonl_preview)

In [ ]:
def estimate_price(description: str, model_choice: str) -> str:
    if not description or not description.strip():
        return "Enter a gadget description."
    p = Product(summary=description.strip(), price=0.0)
    pred = predict_price(p, model_choice)
    return f"Estimated price: ${pred:.2f}"

with gr.Blocks(title="Week 6 — Gadget Pricer") as demo:
    gr.Markdown("## Gadget Pricer — iPhone, Samsung, Huawei, etc.")
    model_dropdown = gr.Dropdown(
        choices=["OpenAI (gpt-4o-mini)", "Anthropic (claude-3-5-sonnet)", "Ollama (llama3.2)"],
        value="OpenAI (gpt-4o-mini)",
        label="Model",
    )
    desc_in = gr.Textbox(label="Gadget description", lines=3, placeholder="e.g. iPhone 15 Pro 256GB, Titanium...")
    btn = gr.Button("Estimate price")
    price_out = gr.Textbox(label="Estimated price")
    btn.click(fn=lambda d, m: estimate_price(d, m), inputs=[desc_in, model_dropdown], outputs=price_out)
    gr.Examples(
        examples=[
            ["iPhone 15 Pro 256GB, Titanium, A17 Pro", "OpenAI (gpt-4o-mini)"],
            ["Samsung Galaxy S24 Ultra 256GB, S Pen", "OpenAI (gpt-4o-mini)"],
            ["Apple AirPods Pro 2nd gen, noise cancellation", "OpenAI (gpt-4o-mini)"],
        ],
        inputs=[desc_in, model_dropdown],
        label="Examples",
    )
demo.launch()